# 🚗 NYC Collision Severity Predictor — Training Notebook

A Decision Support System using **Artificial Neural Network (Multilayer Perceptron)** to predict motor vehicle collision severity in New York City.

**Dataset:** NYC Open Data — Motor Vehicle Collisions (50,000 records)

**Target:** Binary classification → Safe (Class 0) vs Injury/Fatal (Class 1)

---
## STEP 0 — Imports & Configuration

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix

from keras.models import Sequential
from keras.layers import Dense, Dropout, Input, BatchNormalization
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical

# Paths
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(BASE_DIR, 'data')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
CSV_PATH = os.path.join(DATA_DIR, 'NYC_Collisions.csv')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print('Imports loaded ✓')
print(f'Data dir: {DATA_DIR}')
print(f'Models dir: {MODELS_DIR}')

---
## STEP 1 — Data Acquisition

Fetch 50,000 records from the NYC Open Data API (Motor Vehicle Collisions — Crashes).

In [ ]:
API_URL = 'https://data.cityofnewyork.us/resource/h9gi-nx95.json'

if os.path.exists(CSV_PATH):
    print(f'Dataset already exists at: {CSV_PATH}')
    df = pd.read_csv(CSV_PATH)
else:
    print('Downloading dataset from NYC Open Data API...')
    all_records = []
    for offset in range(0, 50000, 5000):
        print(f'  Fetching records {offset} to {offset+5000}...')
        resp = requests.get(API_URL, params={'$limit': 5000, '$offset': offset})
        if resp.status_code != 200: break
        batch = resp.json()
        if not batch: break
        all_records.extend(batch)
    df = pd.DataFrame(all_records)
    df.to_csv(CSV_PATH, index=False)
    print(f'Saved {len(df)} records')

print(f'Dataset shape: {df.shape}')
df.head()

---
## STEP 2 — Feature Engineering & Target Creation

Create the binary target variable and extract time-based features from raw data.

In [ ]:
# Standardize column names
df.columns = df.columns.str.upper().str.replace(' ', '_')
if 'VEHICLE_TYPE_CODE_1' not in df.columns and 'VEHICLE_TYPE_CODE1' in df.columns:
    df.rename(columns={'VEHICLE_TYPE_CODE1': 'VEHICLE_TYPE_CODE_1'}, inplace=True)
if 'VEHICLE_TYPE_CODE_2' not in df.columns and 'VEHICLE_TYPE_CODE2' in df.columns:
    df.rename(columns={'VEHICLE_TYPE_CODE2': 'VEHICLE_TYPE_CODE_2'}, inplace=True)

# Convert numeric columns
df['NUMBER_OF_PERSONS_INJURED'] = pd.to_numeric(df['NUMBER_OF_PERSONS_INJURED'], errors='coerce').fillna(0).astype(int)
df['NUMBER_OF_PERSONS_KILLED'] = pd.to_numeric(df['NUMBER_OF_PERSONS_KILLED'], errors='coerce').fillna(0).astype(int)

# Binary Target
df['SEVERITY_CLASS'] = ((df['NUMBER_OF_PERSONS_KILLED'] > 0) | (df['NUMBER_OF_PERSONS_INJURED'] > 0)).astype(int)

# Extract CRASH_HOUR
def extract_hour(t):
    try: return int(str(t).strip().split(':')[0])
    except: return 12
df['CRASH_HOUR'] = df['CRASH_TIME'].apply(extract_hour)

# Extract DAY_OF_WEEK (0=Monday, 6=Sunday) and MONTH
df['CRASH_DATE'] = pd.to_datetime(df['CRASH_DATE'], errors='coerce')
df['DAY_OF_WEEK'] = df['CRASH_DATE'].dt.dayofweek.fillna(0).astype(int)
df['MONTH'] = df['CRASH_DATE'].dt.month.fillna(1).astype(int)

# === Engineered Features ===
# Binary contextual features
df['IS_WEEKEND'] = (df['DAY_OF_WEEK'] >= 5).astype(int)
df['IS_NIGHT'] = ((df['CRASH_HOUR'] >= 21) | (df['CRASH_HOUR'] <= 5)).astype(int)
df['IS_RUSH_HOUR'] = (((df['CRASH_HOUR'] >= 7) & (df['CRASH_HOUR'] <= 9)) |
                       ((df['CRASH_HOUR'] >= 16) & (df['CRASH_HOUR'] <= 19))).astype(int)

# Cyclic encoding for hour and month (captures circular nature)
df['HOUR_SIN'] = np.sin(2 * np.pi * df['CRASH_HOUR'] / 24)
df['HOUR_COS'] = np.cos(2 * np.pi * df['CRASH_HOUR'] / 24)
df['MONTH_SIN'] = np.sin(2 * np.pi * (df['MONTH'] - 1) / 12)
df['MONTH_COS'] = np.cos(2 * np.pi * (df['MONTH'] - 1) / 12)

# Count vehicles for statistics (NOT model input)
def count_vehicles(row):
    count = 0
    for c in ['VEHICLE_TYPE_CODE_1','VEHICLE_TYPE_CODE_2','VEHICLE_TYPE_CODE_3','VEHICLE_TYPE_CODE_4','VEHICLE_TYPE_CODE_5']:
        if c in row and pd.notnull(row[c]) and str(row[c]).strip() != '': count += 1
    return max(1, count)
df['NUM_VEHICLES'] = df.apply(count_vehicles, axis=1)

print('Target Variable Distribution:')
print(df['SEVERITY_CLASS'].value_counts().sort_index())
print(f'\n  Class 0 (Safe):         {(df["SEVERITY_CLASS"]==0).sum()}')
print(f'  Class 1 (Injury/Fatal): {(df["SEVERITY_CLASS"]==1).sum()}')
print(f'\nEngineered Features: IS_WEEKEND, IS_NIGHT, IS_RUSH_HOUR, HOUR_SIN/COS, MONTH_SIN/COS')

In [ ]:
# Visualize Target + Crash Hour Distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ['#2ecc71', '#e74c3c']
counts = df['SEVERITY_CLASS'].value_counts().sort_index()
axes[0].bar(['Safe (0)', 'Injury/Fatal (1)'], counts.values, color=colors, edgecolor='black')
axes[0].set_title('Target Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 300, f'{v:,}', ha='center', fontweight='bold')

df['CRASH_HOUR'].hist(bins=24, ax=axes[1], color='#3498db', edgecolor='black', alpha=0.8)
axes[1].set_title('Crash Hour Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Hour of Day')

day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
day_counts = df['DAY_OF_WEEK'].value_counts().sort_index()
axes[2].bar(day_names, day_counts.values, color='#9b59b6', edgecolor='black', alpha=0.8)
axes[2].set_title('Crashes by Day of Week', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

---
## STEP 3 — Data Preprocessing

Prepare features for the neural network: StandardScaler for numerical, OneHotEncoder for categorical.

**Features used:**
- Numerical (10): CRASH_HOUR, DAY_OF_WEEK, MONTH, IS_WEEKEND, IS_NIGHT, IS_RUSH_HOUR, HOUR_SIN, HOUR_COS, MONTH_SIN, MONTH_COS
- Categorical (4): BOROUGH, VEHICLE_TYPE_CODE_1, CONTRIBUTING_FACTOR_VEHICLE_1, CONTRIBUTING_FACTOR_VEHICLE_2

In [ ]:
# Define feature columns
numerical_cols = [
    'CRASH_HOUR', 'DAY_OF_WEEK', 'MONTH',
    'IS_WEEKEND', 'IS_NIGHT', 'IS_RUSH_HOUR',
    'HOUR_SIN', 'HOUR_COS', 'MONTH_SIN', 'MONTH_COS',
]

categorical_cols = [
    'BOROUGH',
    'VEHICLE_TYPE_CODE_1',
    'CONTRIBUTING_FACTOR_VEHICLE_1',
    'CONTRIBUTING_FACTOR_VEHICLE_2',
]

target_col = 'SEVERITY_CLASS'
feature_cols = numerical_cols + categorical_cols

df_model = df[feature_cols + [target_col]].copy()

# Handle missing values
for col in numerical_cols:
    df_model[col] = df_model[col].fillna(0)

df_model['BOROUGH'] = df_model['BOROUGH'].fillna('UNKNOWN')
df_model['VEHICLE_TYPE_CODE_1'] = df_model['VEHICLE_TYPE_CODE_1'].fillna('UNKNOWN')
df_model['CONTRIBUTING_FACTOR_VEHICLE_1'] = df_model['CONTRIBUTING_FACTOR_VEHICLE_1'].fillna('UNKNOWN')
df_model['CONTRIBUTING_FACTOR_VEHICLE_2'] = df_model['CONTRIBUTING_FACTOR_VEHICLE_2'].fillna('UNKNOWN')

for col in categorical_cols:
    df_model[col] = df_model[col].astype(str).str.strip().str.upper()

# Build preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

X = preprocessor.fit_transform(df_model[feature_cols])
y = df_model[target_col].values

# Save preprocessor & categories
joblib.dump(preprocessor, os.path.join(MODELS_DIR, 'preprocessor.pkl'))
categories_info = {
    'BOROUGH': list(preprocessor.named_transformers_['cat'].categories_[0]),
    'VEHICLE_TYPE_CODE_1': list(preprocessor.named_transformers_['cat'].categories_[1]),
    'CONTRIBUTING_FACTOR_VEHICLE_1': list(preprocessor.named_transformers_['cat'].categories_[2]),
    'CONTRIBUTING_FACTOR_VEHICLE_2': list(preprocessor.named_transformers_['cat'].categories_[3]),
}
joblib.dump(categories_info, os.path.join(MODELS_DIR, 'categories_info.pkl'))

# Save vehicle stats for API insights
joblib.dump(
    df[['BOROUGH','VEHICLE_TYPE_CODE_1','CONTRIBUTING_FACTOR_VEHICLE_1','NUM_VEHICLES','SEVERITY_CLASS']].copy(),
    os.path.join(MODELS_DIR, 'vehicle_stats.pkl')
)

print(f'X shape after preprocessing: {X.shape}')
print(f'y shape: {y.shape}')

In [ ]:
X_train, X_test, y_train_int, y_test_int = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
y_train = to_categorical(y_train_int, num_classes=2)
y_test  = to_categorical(y_test_int, num_classes=2)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

---
## STEP 4 — Build & Train the ANN (MLP)

Architecture: 256→BN→Dropout(0.4)→128→BN→Dropout(0.4)→64→2(softmax)

**Class Weight Balancing** is applied to address class imbalance (64.5% Safe vs 35.5% Injury/Fatal). This improves Recall on Class 1 without sacrificing overall accuracy.

In [ ]:
# Class weight balancing — moderate weights to maintain accuracy >= 70%
# while improving recall on minority class (Injury/Fatal)
class_weight_dict = {0: 0.9, 1: 1.2}
print(f'Class weights: {class_weight_dict}')

model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dense(2, activation='softmax'),
])
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

history = model.fit(
    X_train, y_train,
    epochs=100, batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    class_weight=class_weight_dict,
    verbose=1,
)

---
## STEP 5 — Evaluation & Visualization

In [ ]:
# Accuracy & Loss Curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].axhline(y=0.70, color='r', linestyle='--', alpha=0.5, label='Target 70%')
axes[0].set_title('Model Accuracy per Epoch', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_title('Model Loss per Epoch', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'\n{"="*50}')
print(f'  TEST ACCURACY : {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'  TEST LOSS     : {test_loss:.4f}')
print(f'{"="*50}')
print('\n✅ TARGET ACHIEVED!' if test_acc >= 0.70 else f'\n⚠️ Below target: {test_acc*100:.2f}%')

In [ ]:
y_pred_proba = model.predict(X_test)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = np.argmax(y_test, axis=1)
class_names = ['Safe (Class 0)', 'Injury/Fatal (Class 1)']
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax, annot_kws={'size': 16, 'fontweight': 'bold'})
ax.set_title('Confusion Matrix', fontsize=16, fontweight='bold')
ax.set_xlabel('Predicted Label', fontsize=12); ax.set_ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Probability Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(y_pred_proba[y_true==0][:,1], bins=50, color='#2ecc71', alpha=0.7, edgecolor='black')
axes[0].axvline(x=0.5, color='red', linestyle='--', linewidth=2)
axes[0].set_title('P(Injury) — Actual Safe', fontsize=12, fontweight='bold')
axes[1].hist(y_pred_proba[y_true==1][:,1], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[1].axvline(x=0.5, color='red', linestyle='--', linewidth=2)
axes[1].set_title('P(Injury) — Actual Injury/Fatal', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## STEP 6 — Export Model

In [ ]:
model.save(os.path.join(MODELS_DIR, 'severity_model.h5'))
model.save(os.path.join(MODELS_DIR, 'severity_model.keras'))
print('Exported to:', MODELS_DIR)
print('  - severity_model.h5 / .keras')
print('  - preprocessor.pkl')
print('  - categories_info.pkl')
print('  - vehicle_stats.pkl')
print('\n✅ TRAINING COMPLETE!')

---
## STEP 7 — Sample Prediction (Demo)

In [ ]:
sample = pd.DataFrame([{
    'CRASH_HOUR': 23, 'DAY_OF_WEEK': 5, 'MONTH': 12,
    'IS_WEEKEND': 1, 'IS_NIGHT': 1, 'IS_RUSH_HOUR': 0,
    'HOUR_SIN': float(np.sin(2 * np.pi * 23 / 24)),
    'HOUR_COS': float(np.cos(2 * np.pi * 23 / 24)),
    'MONTH_SIN': float(np.sin(2 * np.pi * 11 / 12)),
    'MONTH_COS': float(np.cos(2 * np.pi * 11 / 12)),
    'BOROUGH': 'MANHATTAN',
    'VEHICLE_TYPE_CODE_1': 'SEDAN',
    'CONTRIBUTING_FACTOR_VEHICLE_1': 'ALCOHOL INVOLVEMENT',
    'CONTRIBUTING_FACTOR_VEHICLE_2': 'UNKNOWN',
}])
pred = model.predict(preprocessor.transform(sample), verbose=0)
print(f'Input: Saturday night, Dec, Manhattan, Sedan, Alcohol')
print(f'  Safe:         {pred[0][0]*100:.2f}%')
print(f'  Injury/Fatal: {pred[0][1]*100:.2f}%')
print(f'  Predicted: {"✅ Safe" if np.argmax(pred[0]) == 0 else "🚨 Injury/Fatal"}')